# Этап 8: Сравнение моделей

Сводное сравнение всех 5 моделей (Logistic Regression, CatBoost, MLP, FT-Transformer, TabM).
**Основной вопрос:** способен ли TabM превзойти CatBoost?

Ноутбук только читает артефакты из `results/` — модели не переобучаются.

## 1. Загрузка артефактов

In [ ]:
# Импортируем библиотеки для работы с данными и построения графиков
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import roc_curve, auc  # roc_curve строит ROC, auc считает площадь под ней

# Папка с артефактами (метриками, предсказаниями, графиками)
RESULTS = Path('../results')

# Список 5 моделей: (ключ для файла, человекочитаемое название)
MODELS = [
    ('logreg',      'Logistic Regression'),   # Линейная модель-бейзлайн
    ('catboost',    'CatBoost'),              # Градиентный бустинг
    ('mlp',         'MLP'),                  # Многослойный перцептрон (нейросеть)
    ('transformer', 'FT-Transformer'),        # Трансформер для табличных данных
    ('tabm',        'TabM'),                  # Новая архитектура: ансамбль линейных голов
]

## 2. Загрузка метрик

In [ ]:
def load_metrics(model_key: str) -> dict:
    """Загрузить метрики модели в едином F2-only формате."""
    with open(RESULTS / "metrics" / f"{model_key}.json", encoding="utf-8") as f:
        raw = json.load(f)

    splits = raw.get("metrics", raw)
    test = raw.get("test_f2_metrics") or splits.get("test", {})

    return {
        "train": splits.get("train", {}),
        "val": splits.get("val", {}),
        "test": test,
        "threshold_f2": raw.get("threshold_f2") or raw.get("threshold", 0.0),
    }


all_metrics = {key: load_metrics(key) for key, _ in MODELS}
print("Metrics loaded for:", list(all_metrics.keys()))

## 3. Итоговая таблица — F2-оптимальный порог

In [ ]:
# Список метрик для отображения в итоговой таблице
METRIC_COLS = ['roc_auc', 'precision', 'recall', 'f2']
METRIC_NAMES = ['ROC-AUC', 'Precision', 'Recall', 'F2']

rows = []
for key, label in MODELS:
    m = all_metrics[key]['test']  # Берём метрики на ТЕСТОВОЙ выборке
    row = {
        'Model': label,
        # Порог классификации, подобранный по максимальному F2 на val-выборке
        'Threshold': round(all_metrics[key]['threshold_f2'], 4),
    }
    for col, name in zip(METRIC_COLS, METRIC_NAMES):
        row[name] = round(m.get(col, float('nan')), 4)
    # FP = False Positives: сколько здоровых пациентов модель ошибочно пометила как "группа риска"
    row['FP (test)'] = m.get('fp', '?')
    rows.append(row)

df_test = pd.DataFrame(rows).set_index('Model')
print('=== Test-set метрики (F2-оптимальный порог) ===')
# ROC-AUC: качество ранжирования независимо от порога (главная метрика для сравнения моделей)
# Precision: из всех "предсказанных рисковых" — какая доля реально вернулась (точность)
# Recall: из всех реально рисковых — сколько мы нашли (полнота, НЕ ПРОПУСТИТЬ важно)
# F2: взвешенное среднее precision и recall, где recall весит в 2 раза больше
# FP: абсолютное число ложных тревог — важно для операционных расходов
df_test

## 4. Сравнение метрик по моделям


In [ ]:
# Сравниваем все модели по основным test-метрикам.
metric_cols = ["ROC-AUC", "Precision", "Recall", "F2"]
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.ravel()

for ax, metric in zip(axes, metric_cols):
    order = df_test[metric].sort_values(ascending=False).index
    sns.barplot(
        data=df_test.reset_index(),
        x="Model",
        y=metric,
        order=order,
        ax=ax,
        color="steelblue",
    )
    ax.set_title(metric)
    ax.set_xlabel("")
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=20)
    ax.bar_label(ax.containers[0], fmt="%.3f", fontsize=9, padding=2)

plt.tight_layout()
plt.savefig(RESULTS / "metrics_comparison_bars.png", dpi=120)
plt.show()

# FP показываем отдельно, потому что это абсолютное число, а не доля.
fp_order = df_test["FP (test)"].sort_values().index
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(
    data=df_test.reset_index(),
    x="Model",
    y="FP (test)",
    order=fp_order,
    ax=ax,
    color="coral",
)
ax.set_title("False Positives на test")
ax.set_xlabel("")
ax.set_ylabel("FP")
ax.tick_params(axis="x", rotation=20)
ax.bar_label(ax.containers[0], fmt="%.0f", fontsize=9, padding=2)
plt.tight_layout()
plt.savefig(RESULTS / "fp_comparison_bar.png", dpi=120)
plt.show()


## 5. ROC-кривые

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for key, label in MODELS:
    # Загружаем предсказания модели на тестовой выборке (y_true и y_proba)
    pred = pd.read_csv(RESULTS / 'predictions' / f'{key}_test.csv')
    # roc_curve вычисляет TPR и FPR при разных порогах
    fpr, tpr, _ = roc_curve(pred['y_true'], pred['y_proba'])
    roc_auc = auc(fpr, tpr)  # Площадь под кривой (AUC)
    ax.plot(fpr, tpr, label=f'{label} (AUC={roc_auc:.3f})', linewidth=1.5)

# Диагональная линия — случайная модель (AUC = 0.5)
ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, alpha=0.5, label='Случайная')
ax.set_xlabel('False Positive Rate')  # Доля здоровых, ошибочно помеченных как больные
ax.set_ylabel('True Positive Rate')   # Доля больных, правильно найденных (recall)
ax.set_title('ROC-кривые (тестовая выборка)')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig(RESULTS / 'roc_curves.png', dpi=120)
plt.show()
# Все кривые кластеризуются вокруг AUC ≈ 0.65-0.66 — это потолок датасета, а не проблема моделей.
print("Все модели кластеризуются вокруг AUC≈0.65-0.66 — потолок датасета")

## 6. Выводы и честный вердикт

### 1. Потолок ROC-AUC около 0.66 сохраняется

SHAP-отбор признаков (RandomForest + 3-fold CV на LogReg, варианты top-10/12/15/20/25/30/all из ~37 кандидатов) показал, что лучший набор — top-25 с ROC-AUC около 0.644 на CV, а различия между наборами признаков меньше 0.002.

На test-выборке все модели также остаются в узком диапазоне ROC-AUC 0.651–0.659. Значит, основной потолок качества задаётся данными и постановкой задачи, а не только выбранной архитектурой.

### 2. Реальный рычаг — выбор операционной точки

Большое число false positives — это не ошибка расчёта, а следствие F2-оптимального порога: в основном протоколе recall важнее precision, поэтому модели сознательно помечают больше пациентов как группу риска.

Основной протокол оставлен на F2, потому что для задачи раннего выявления важнее высокий recall и низкое число FN, то есть меньше пропущенных реадмиссий.

### 3. Сравнение всех моделей на test

Все модели сравниваются на F2-оптимальном пороге. Для FP и FN меньше — лучше.

| Модель | ROC-AUC | Precision | Recall | F2 | FP | FN |
|---|---:|---:|---:|---:|---:|---:|
| LogReg | 0.6529 | 0.1162 | 0.7514 | 0.3590 | 7 169 | 312 |
| MLP | 0.6507 | **0.1193** | 0.7155 | 0.3578 | **6 631** | 357 |
| Transformer | 0.6580 | 0.1083 | **0.8359** | 0.3567 | 8 634 | **206** |
| CatBoost | **0.6586** | 0.1171 | 0.7498 | 0.3604 | 7 094 | 314 |
| TabM | 0.6574 | 0.1121 | 0.8175 | **0.3621** | 8 123 | 229 |

### 4. Отдельно TabM vs CatBoost

| Метрика | CatBoost | TabM | Кто лучше |
|---|---:|---:|---|
| ROC-AUC | **0.6586** | 0.6574 | CatBoost |
| Precision | **0.1171** | 0.1121 | CatBoost |
| Recall | 0.7498 | **0.8175** | TabM |
| F2 | 0.3604 | **0.3621** | TabM |
| FP | **7 094** | 8 123 | CatBoost |
| FN | 314 | **229** | TabM |

**TabM выиграла у CatBoost по основной целевой метрике F2**: 0.3621 против 0.3604. Также TabM лучше по recall и FN: она находит больше реадмиссий и пропускает 229 пациентов вместо 314.

**CatBoost выиграл у TabM по ROC-AUC, precision и FP**: он чуть лучше ранжирует риск, точнее среди положительных предсказаний и даёт меньше ложных тревог — 7 094 FP против 8 123 у TabM.

**Вывод по паре TabM/CatBoost:** если ориентироваться на выбранный F2-протокол, победила TabM. Но преимущество небольшое, а цена победы — больше ложных тревог. Если важнее снизить FP и сохранить чуть лучший ROC-AUC, CatBoost выглядит практичнее. Поэтому TabM выигрывает именно как recall-oriented модель, а CatBoost остаётся более сбалансированным вариантом.

### 5. Кто выиграл и кто проиграл среди всех моделей

- **TabM выиграл по основной целевой метрике F2**: 0.3621 против 0.3604 у CatBoost, 0.3590 у LogReg, 0.3578 у MLP и 0.3567 у Transformer. Разрыв небольшой, но по выбранному протоколу это лучший результат.
- **CatBoost выиграл по ROC-AUC**: 0.6586. Это лучший результат ранжирования риска, но преимущество над Transformer (0.6580) и TabM (0.6574) минимальное.
- **Transformer выиграл по recall и FN**: recall 0.8359 и всего 206 FN. При этом он проиграл по precision и FP: precision 0.1083 и 8 634 ложных тревоги.
- **MLP выиграл по precision и FP**: precision 0.1193 и 6 631 FP. Но он проиграл по recall и FN: recall 0.7155 и 357 FN, поэтому хуже подходит под F2-ориентированную задачу.
- **LogReg осталась сильным базовым уровнем**: она почти не уступает сложным моделям по F2, но нигде не стала лидером.

**Итоговый вердикт:** по основному протоколу победила TabM, потому что у неё лучший F2. Если важнее ранжирование риска без привязки к порогу, выигрывает CatBoost. Если важнее минимизировать пропущенные реадмиссии, выигрывает Transformer, но ценой максимального числа ложных тревог. Поэтому TabM можно считать лучшей моделью именно для выбранной F2-постановки, а не абсолютным победителем по всем критериям.
